# L2b: Test and Strengthen a Fibonacci Function

L2a introduced Fibonacci as an example of a function with a documented interface. In this lab, we test that calculation at its numerical boundary, implement a version that prevents silent integer overflow, and write regression tests for its complete interface.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Identify a silent numerical failure:__ Compare a computed value with a reference value and explain why successful execution does not establish numerical correctness.
> * __Implement a defensive numerical interface:__ Document the return representation, validate the input, and prevent an `Int64` calculation from continuing beyond its supported range.
> * __Test the complete function contract:__ Use Julia's `Test` standard library to verify ordinary cases, boundary cases, and expected errors.

Let's get started!
___

## Setup, Data, and Prerequisites

The setup file activates the course environment, loads the student implementation from [`src/Compute.jl`](src/Compute.jl), and imports the packages used in this lab.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The setup loads [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/), which provides the testing macros used throughout this lab, and the `L2bFibonacci` module from [`src/Compute.jl`](src/Compute.jl), which exports the `fibonacci_sequence(...)` function you will complete in Task 2.

___

## Task 1: Test the original Fibonacci calculation

The `fibonacci_unchecked(n)` function uses the iterative calculation introduced in L2a. It accepts a nonnegative index and returns a vector containing $F_0$ through $F_n$. The vector stores `Int64` values, but the function does not check whether every requested Fibonacci number fits in that type.

We will test an ordinary case and then inspect the first value outside the `Int64` range.

In [ ]:
"""
    fibonacci_unchecked(n::Int64) -> Vector{Int64}

Return the Fibonacci values from F_0 through F_n without checking whether
the requested values fit in Int64.
"""
function fibonacci_unchecked(n::Int64)::Vector{Int64}
    # Reject negative indices before allocating the result vector.
    @assert n >= 0 "n must be nonnegative"

    # Allocate one position for every value from F_0 through F_n.
    sequence = Vector{Int64}(undef, n + 1)
    sequence[1] = 0
    n == 0 && return sequence

    # Store F_1 and compute each remaining value from its two predecessors.
    sequence[2] = 1
    for position in 3:length(sequence)
        sequence[position] = sequence[position - 1] + sequence[position - 2]
    end

    return sequence
end

With the function defined, let's evaluate the ordinary case, the boundary case, and the type limit:

In [ ]:
# F_10 is an ordinary reference case that fits comfortably in Int64.
observed_F10 = last(fibonacci_unchecked(10))
expected_F10 = 55

# F_93 is larger than typemax(Int64). BigInt stores the exact reference value.
observed_F93 = last(fibonacci_unchecked(93))
expected_F93 = big"12200160415121876738"

(
    ordinary_case = (observed = observed_F10, expected = expected_F10),
    boundary_failure = (observed = observed_F93, expected = expected_F93),
    largest_Int64 = typemax(Int64),
)

The ordinary case is correct. The call for $F_{93}$ also finishes, but its result is incorrect because the exact value exceeds the largest `Int64` value, which [the `typemax(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.typemax) returns. Ordinary integer addition wraps when the result cannot be represented, so the program returns a value that does not satisfy the function's stated purpose.

A reference case establishes the failure. The type limit identifies the cause. The corrected interface will stop at $F_{92}$, the largest Fibonacci number that fits in an `Int64`.

In [ ]:
@testset "original Fibonacci calculation" begin
    # Confirm the ordinary reference case.
    @test observed_F10 == expected_F10

    # Record the known boundary failure before correcting the interface.
    @test observed_F93 != expected_F93
end

___

## Task 2: Implement input validation and checked arithmetic

Task 1 showed that the unchecked calculation fails silently: it returns a wrong value instead of raising an error. The corrected interface must validate its input and refuse any request whose exact result cannot be represented. Open [`src/Compute.jl`](src/Compute.jl) and complete its three `TODO` sections. The public [`fibonacci_sequence(...)` function](src/Compute.jl) has the following contract:

> * `fibonacci_sequence(n)` returns a `Vector{Int64}` containing $F_0$ through $F_n$. Because Julia arrays use one-based indexing, $F_n$ is stored at position `n + 1`.
> * The supported range is `0 <= n <= 92`. The function rejects negative indices and indices above 92 with an `ArgumentError`.
> * `Bool` is a subtype of `Integer` in Julia, but `true` and `false` are not sequence indices for this interface. The function rejects them explicitly.
> * The calculation uses [the `Base.Checked.checked_add(...)` function](https://docs.julialang.org/en/v1/base/math/#Base.Checked.checked_add), which raises an [`OverflowError`](https://docs.julialang.org/en/v1/base/base/#Core.OverflowError) if an addition exceeds the `Int64` range.

The upper-bound check should prevent that overflow from occurring during a valid call. Checked addition provides a second check inside the calculation.

After completing the source file, restart the kernel and run the notebook from the beginning so that Julia loads the revised module. Until all three `TODO` sections are complete, calling `fibonacci_sequence(...)` raises a direct implementation error.

In [ ]:
# Compute an ordinary sequence with the completed defensive implementation.
sequence = fibonacci_sequence(10)
(sequence = sequence, F10 = sequence[10 + 1])

### Supported and unsupported boundaries

The calls `fibonacci_sequence(0)` and `fibonacci_sequence(1)` are the smallest supported requests. Each must return a complete vector of the documented length. At the other end of the contract, $F_{92}$ is supported and $F_{93}$ is rejected before the calculation begins.

In [ ]:
# Evaluate both lower-bound cases and the largest supported result.
base_cases = (n0 = fibonacci_sequence(0), n1 = fibonacci_sequence(1))
largest_supported = last(fibonacci_sequence(92))

# Catch one rejected call so that the error message can be inspected.
upper_boundary_message = try
    fibonacci_sequence(93)
    "no error"
catch error
    sprint(showerror, error)
end

(base_cases = base_cases, F92 = largest_supported, F93 = upper_boundary_message)

___

## Task 3: Test the complete interface

Julia's [`Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) provides macros for grouping tests and stating expected behavior. These tests become regression tests: they record behavior that must remain correct after the implementation changes.

| Test tool | Purpose in this lab |
|:--|:--|
| [`@testset`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@testset) | Groups the Fibonacci checks and prints one summary. |
| [`@test`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test) | Verifies returned values, vector length, and the upper boundary. |
| [`@test_throws`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test_throws) | Verifies that invalid inputs raise the documented exception type. |

The test set covers the two smallest supported inputs, an ordinary input, the largest supported input, and every input category rejected by the interface.

In [ ]:
@testset "defensive Fibonacci interface" begin
    # Verify the smallest supported inputs.
    @test fibonacci_sequence(0) == [0]
    @test fibonacci_sequence(1) == [0, 1]

    # Verify an ordinary sequence and its documented indexing convention.
    @test fibonacci_sequence(10) == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]
    @test length(fibonacci_sequence(10)) == 11

    # Verify the largest value supported by the Int64 return type.
    @test last(fibonacci_sequence(92)) == 7_540_113_804_746_346_429

    # Verify every invalid-input category in the public contract.
    @test_throws ArgumentError fibonacci_sequence(-1)
    @test_throws ArgumentError fibonacci_sequence(93)
    @test_throws ArgumentError fibonacci_sequence(2.5)
    @test_throws ArgumentError fibonacci_sequence(true)
end

### Optional Python implementation

The optional Python implementation in [`src/fibonacci.py`](src/fibonacci.py) uses the same input range and return representation. Python integers do not overflow at $F_{93}$, so the implementation enforces the upper bound explicitly to preserve the shared contract. It also rejects `bool`, which is a subclass of `int` in Python.

The implementation and its tests include comments describing the function and the key steps. Let's display the source file:

In [ ]:
python_source = joinpath(CHEME5800_L2B_ROOT, "src", "fibonacci.py")
print(read(python_source, String))

Run the Python regression tests from the repository root:

```bash
python -m unittest discover -s weeks/week-02/L2b/src -p 'test_*.py'
```

___

## Summary

In this lab, we identified silent integer overflow in a Fibonacci calculation, implemented a guarded `Int64` interface, and wrote regression tests for its supported and unsupported inputs.

> __Key Takeaways:__
>
> * **Successful execution does not establish correctness:** The original calculation returned a value beyond its supported range without raising an error, but comparison with an exact reference value showed that the result was incorrect.
> * **The return type can set a numerical boundary:** Because the function returns `Vector{Int64}`, its documented input range stops at the largest Fibonacci number an `Int64` can hold and rejects requests that the return type cannot represent.
> * **Regression tests define the complete interface:** Tests for ordinary values, boundary values, and expected errors record the behavior that later changes must preserve.

We will use the same combination of reference cases, explicit numerical limits, input validation, and regression tests when building larger numerical programs.

___